# Quickstart: nuclear outlines in 2D

Finds the outline of every nucleus in z-stacks of a staining that also labels the nuclear envelope, such as an ER
marker or a lamin: a section through a nucleus is then a bright ring around a dark inside. For every stack the
notebook writes to `OUT_DIR`

- `<name>_labels.tif`: a label image (0 = background, 1, 2, ... = nuclei, largest first) with the pixel size,
- `<name>_nuclei.csv`: area, centroid, shape and z window per nucleus,
- `<name>_parameters.json`: the parameters, voxel size and versions, to repeat the run,
- `<name>_overview.png`, `<name>_gallery.png`: the outlines on the image, to check the result.

How it works, and how to tune the parameters: `2_method_2d.ipynb`. Volumes in 3D: `3_quickstart_3d.ipynb`.

**Data.** Put your TIFF z-stacks in `data/` next to this notebook, or set `DATA_DIR` to their folder. Without
any, the notebook writes and uses a small synthetic stack (`example_data/synthetic.tif`), which shows how the pipeline
works but is no substitute for real images. See the README for the input the pipeline expects.

**Start with one stack** (under a minute each): uncomment `paths = paths[:1]` below, check the result and adjust the
parameters, then run all stacks.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

import nucseg

In [ ]:
DATA_DIR = Path("data")      # folder with your TIFF z-stacks
OUT_DIR = Path("results")
CHANNEL = None               # channel of multichannel files (0-based); None for single-channel files
UM_PER_PIXEL = None          # pixel size in µm; None reads it from the ImageJ/OME metadata

paths = nucseg.find_stacks([DATA_DIR]) if DATA_DIR.exists() else []
if not paths:
    print(f"No TIFFs in {DATA_DIR}/, using the synthetic example instead.")
    paths = [nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")]
# paths = paths[:1]          # try one stack first
[p.name for p in paths]

## Parameters

| | |
|---|---|
| **read from the file** | pixel size (ImageJ/OME metadata), printed for every stack; set `UM_PER_PIXEL` above if it is missing or wrong |
| **automatic** | intensity scaling (each projection is scaled to its own 99.8th percentile), the projection each nucleus is taken from |
| **set for your data** | `window`: slices averaged per projection, about 1 µm (3 at 0.3 µm slices) |
| | `min_area_um2`, `max_area_um2`, `min_width_um`: the size of your nuclei |
| | `border_frame`: also find the visible part of nuclei cut by the image edge |
| **speed** | `bin_factor`: pixels averaged per axis; higher is faster, but too coarse loses thin rings |
| | `z_first`, `z_last`: the slices through the nuclei (default: all slices) |
| **tuning only** | everything else; see the table *Tuning* at the end of `2_method_2d.ipynb` |

All fields are documented in `help(nucseg.Parameters)`.

In [ ]:
parameters = nucseg.Parameters(
    bin_factor=4,          # pixels averaged per axis, for speed; too coarse loses thin rings
    window=3,              # slices averaged per projection, about 1 µm
    min_area_um2=25.0,     # size of your nuclei
    max_area_um2=800.0,
    min_width_um=5.0,
    z_first=None,          # first and last slice used (inclusive); None = first / last
    z_last=None,
    border_frame=True,     # also nuclei cut by the image edge, marked border=True (default False)
)

## Segmentation

About 35-40 s per 2048 x 2048 x 20 stack with `bin_factor=4` (Apple M4 Pro), about half that with slices 4-14.

Check the overview (outlines on the image) and the gallery (every nucleus close up) of each stack. If nuclei are
missing, split or merged, or outlines run into the ER, the table *Tuning* at the end of `2_method_2d.ipynb` lists the
parameter to change for each symptom.

In [ ]:
tables = []
for path in paths:
    stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
    z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
    print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
          f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}; segmenting ...")
    segmentation = nucseg.segment_2d(stack, parameters)
    nucseg.save_results(OUT_DIR, stack, segmentation)
    tables.append(pd.DataFrame(segmentation.table()).assign(image=stack.name))
    print(f"  {len(segmentation.nuclei)} nuclei")
    display(Image(OUT_DIR / f"{stack.name}_overview.png"), Image(OUT_DIR / f"{stack.name}_gallery.png"))

## Nuclei

One row per nucleus; `label` is the value in the label image. Leave nuclei with `border = True` out of size
statistics: their area is only the visible part.

In [ ]:
nuclei = pd.concat(tables, ignore_index=True).set_index(["image", "label"])
nuclei

In [ ]:
nuclei[~nuclei.border].groupby("image")["area_um2"].describe().round(1)